# ML Project · Рабочая версия

Пять связанных экспериментов на реальных данных: подготовка, выбор модели по validation, оценка на test, интерпретация ошибок.

Это новая адаптация тем курса, подготовленная с помощью Codex. Исходные задания ВШЭ не представлены как полностью восстановленные; отсутствующие Taxi/Autos и другие данные заменены явно документированными наборами. Старые решения не входят в исполняемый код.

1. [Регрессия и EDA](01-regression.ipynb)
2. [Градиентный спуск](02-optimization.ipynb)
3. [Классификация и калибровка](03-classification.ipynb)
4. [Деревья решений](04-trees.ipynb)
5. [Градиентный бустинг](05-boosting.ipynb)

Для запуска всех ячеек не нужны внешние CSV, Kaggle или доступ к сети после установки зависимостей.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
while not (ROOT / 'mlproject').is_dir():
    if ROOT.parent == ROOT:
        raise RuntimeError('Open this notebook inside the project folder')
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
from IPython.display import display, Image
from mlproject.data import dataset, split
from mlproject.experiments import run


## Данные и воспроизводимость

Diabetes: регрессия количественного показателя прогрессирования заболевания. Wisconsin Diagnostic Breast Cancer: бинарная классификация; **1 = malignant**. Это учебные эксперименты, не медицинская система. Входные данные встроены в scikit-learn.

In [2]:
rows=[]
for task in ['regression','classification']:
    X,y=dataset(task)
    sizes={key:len(pair[1]) for key,pair in split(task).items()}
    rows.append(dict(task=task,objects=len(y),features=X.shape[1],missing=int(X.isna().sum().sum()),**sizes))
display(pd.DataFrame(rows))

,task,objects,features,missing,train,val,test
0,regression,442,10,0,264,89,89
1,classification,569,30,0,341,114,114


## Протокол

- Один фиксированный seed 42; разбиение примерно 60/20/20, стратификация для классификации.
- Масштабирование обучается только на train. В калибровке scaler находится внутри CV-estimator.
- Validation выбирает параметры. Test не используется для настройки.
- Baseline и все метрики сохраняются в reports. Один split не даёт оценки статистической устойчивости.
- Собственные реализации: GD/SGD/Momentum/Adam, бинарный CART и логистический бустинг. Для слабых моделей бустинга используется DecisionTreeRegressor из sklearn.

Команда полной проверки: `python scripts/verify.py`. Она проверяет вычисления тестами и запускает все 6 ноутбуков отдельными чистыми ядрами. В GitHub Actions можно выполнить её через браузер.